# Lesson 08: Logistic Regression, Confusion Matrices & ROC Curves

Encode ordinal (`Item Size`) and nominal (`City Name`, `Package`, `Variety`, `Origin`) pumpkin features with `ColumnTransformer`, train `LogisticRegression`, and plot the ROC curve.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

rng = np.random.default_rng(12)
n = 300
sizes = rng.choice(['sml', 'med', 'med-lge', 'lge', 'xlge'], size=n)
varieties = rng.choice(['MINIATURE', 'PIE TYPE', 'FAIRYTALE'], size=n)
cities = rng.choice(['BALTIMORE', 'CHICAGO', 'BOSTON'], size=n)
is_white = ((varieties == 'MINIATURE') & np.isin(sizes, ['sml', 'med'])).astype(int)
flip = rng.random(n) < 0.08
colors = np.where(np.logical_xor(is_white, flip), 'WHITE', 'ORANGE')

pumpkins = pd.DataFrame({
    'City Name': cities,
    'Package': np.where(varieties == 'MINIATURE', '1/2 bushel', '1 1/9 bushel'),
    'Variety': varieties,
    'Origin': np.where(cities == 'BOSTON', 'MASSACHUSETTS', 'MARYLAND'),
    'Item Size': sizes,
    'Color': colors,
})

ord_enc = OrdinalEncoder(categories=[['sml', 'med', 'med-lge', 'lge', 'xlge']])
cat_enc = OneHotEncoder(sparse_output=False)
ct = ColumnTransformer([
    ('ord', ord_enc, ['Item Size']),
    ('cat', cat_enc, ['City Name', 'Package', 'Variety', 'Origin']),
])
ct.set_output(transform='pandas')
X = ct.fit_transform(pumpkins)
y = LabelEncoder().fit_transform(pumpkins['Color'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
model = LogisticRegression(max_iter=500).fit(X_train, y_train)
preds = model.predict(X_test)
print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds))

y_scores = model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_scores)
print("ROC AUC:", round(roc_auc_score(y_test, y_scores), 4))
plt.plot([0, 1], [0, 1], 'k--')
plt.plot(fpr, tpr, color='#A7452A', linewidth=2)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Pumpkin Color ROC Curve')
plt.show()

## Assignment: Retrying Classification

Compare `LogisticRegression(C=0.1)` vs `LogisticRegression(C=10.0)` and check how regularization `C` shifts Precision, Recall, and ROC-AUC.